# cv_r1 話者監査ノート — 指定話者の元音声をまとめて聴く

用途: 性別ラベルと聴感の照合、マップ上で気になった話者の確認、近傍話者との聴き比べ。

**Colab**（eval 等でデータ配置済みのセッション）でも **Mac**（パイプライン出力ツリー）でも動く。
§1 でパスを合わせ、§2 で1話者を聴取、§3 で（speaker_map があれば）近傍話者と比較する。

In [ ]:
# ===== §0 環境ガード（Mac は sbv2 カーネル推奨 / Colab はそのままで可）=====
import sys
from importlib.util import find_spec
IS_COLAB = find_spec("google.colab") is not None
if not IS_COLAB and "sbv2" not in sys.prefix:
    print("★Mac では Python (sbv2) カーネル推奨（soundfile が要る）。現在:", sys.prefix)
for lib in ("numpy", "soundfile", "IPython"):
    assert find_spec(lib), f"★{lib} が無い"
print("環境 OK:", "Colab" if IS_COLAB else sys.prefix)


In [ ]:
# ===== §1 設定とデータ読み込み =====
from pathlib import Path
import json, collections

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# --- データ所在の自動検出（見つかった最初のものを使う）---
_CANDIDATES = ([Path("/content/Style-Bert-VITS2/Data/cv_r1"),      # eval/train セッション（配置済み）
                Path("/content/audit_data/Data/cv_r1")]            # §1.5 で選択展開した場合
               if IS_COLAB else
               [Path("/Users/slp/tmp/synth/sbv2finetune/cv_r1/sbv2_data/cv_r1")])  # Mac prep 出力
DATA = next((d for d in _CANDIDATES if (d / "esd_train.list").exists() or (d / "esd.list").exists()), None)

_MA = Path("/content/drive/MyDrive/Style-Bert-VITS2/model_assets") if IS_COLAB else None
MAP  = _MA / "speaker_map.json"  if _MA else None   # 任意（§3 と性別表示に使用）
META = _MA / "speaker_meta.json" if _MA else None

SPK = collections.defaultdict(list)
if DATA is None:
    print("★データ未配置。次の §1.5 を実行して監査対象話者ぶんだけ取得する（全量展開は不要）")
else:
    def _resolve(p):
        q = Path(p)
        if q.is_absolute() and q.exists(): return q
        if "Data/cv_r1/" in p:
            r = DATA / p.split("Data/cv_r1/", 1)[1]
            if r.exists(): return r
        r = DATA / "wavs" / q.name
        return r if r.exists() else None
    for name in ("esd_train.list", "esd_val.list", "esd.list"):
        f = DATA / name
        if not f.exists(): continue
        for line in open(f, encoding="utf-8"):
            c = line.rstrip("\n").split("|")
            w = _resolve(c[0])
            if w and all(w != x[0] for x in SPK[c[1]]):
                SPK[c[1]].append((w, c[3] if len(c) > 3 else ""))
COORD = json.load(open(MAP, encoding="utf-8")) if (MAP and MAP.exists()) else {}
COORD.pop("_meta", None)
GEN = json.load(open(META, encoding="utf-8")) if (META and META.exists()) else {}
print(f"DATA: {DATA} / 話者 {len(SPK)} / 発話 {sum(map(len, SPK.values()))}"
      f" / map: {bool(COORD)} / 性別メタ: {bool(GEN)}")


In [ ]:
# ===== §1.5（データ未配置のセッション用）対象話者の wav だけ取得 =====
# meta tgz（32MB）から esd を取り、監査したい話者の wav だけを wavs tar から選択展開する。
# wavs tar は Drive の zenodo_cache にあればそれを使い、無ければダウンロード（4.81GB, aria2）。
AUDIT_SPEAKERS = ["cv_0127"]   # ← 監査したい話者（複数可。後から増えたら再実行 = 追加展開）

import subprocess, tarfile
from pathlib import Path
if DATA is not None and all(SPK.get(s) for s in AUDIT_SPEAKERS):
    print("対象話者は取得済み → このセルは何もしない。§1 を再実行してから §2 へ")
else:
    ROOT = Path("/content/audit_data"); ROOT.mkdir(exist_ok=True)
    CACHE = Path("/content/drive/MyDrive/Style-Bert-VITS2/zenodo_cache")
    META_N, WAVS_N = "cadence_cv_r1_meta_v20260702.tgz", "cadence_cv_r1_wavs_v20260702.tar"
    ZURL = "https://zenodo.org/records/21119791/files/{}?download=1"

    def fetch(name, size_hint):
        c = CACHE / name
        if c.exists() and c.stat().st_size > size_hint:
            print(f"{name}: Drive キャッシュを使用"); return c
        dst = ROOT / name
        if not (dst.exists() and dst.stat().st_size > size_hint):
            print(f"{name}: ダウンロード...")
            subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
            subprocess.run(["aria2c", "-x16", "-s16", "-q", "-d", str(ROOT), "-o", name,
                            ZURL.format(name)], check=True)
        return dst

    mt = fetch(META_N, 20_000_000)
    with tarfile.open(mt) as t:
        t.extractall(ROOT)   # esd/config/cadseq（小さい）
    esd = ROOT / "Data/cv_r1/esd_train.list"
    want, targets = set(AUDIT_SPEAKERS), []
    for name in ("esd_train.list", "esd_val.list"):
        f = ROOT / "Data/cv_r1" / name
        if not f.exists(): continue
        for line in open(f, encoding="utf-8"):
            c = line.rstrip("\n").split("|")
            if c[1] in want and "Data/cv_r1/" in c[0]:
                targets.append(c[0][c[0].index("Data/cv_r1/"):])
    print(f"対象 wav: {len(targets)} 本（{', '.join(AUDIT_SPEAKERS)}）")

    wt = fetch(WAVS_N, 4_000_000_000)
    have = {str(x) for x in (ROOT / "Data/cv_r1/wavs").glob("*.wav")} if (ROOT/"Data/cv_r1/wavs").exists() else set()
    need = [t for t in targets if not (ROOT / t).exists()]
    if need:
        print(f"tar から {len(need)} 本を選択展開（tar 全体を走査するため数分）...")
        with tarfile.open(wt) as t:
            nd = set(need)
            for m in t:
                if m.name in nd:
                    t.extract(m, ROOT)
                    nd.discard(m.name)
                    if not nd: break
        print("展開完了")
    print("→ §1 を再実行して読み直し、§2 へ")


In [ ]:
# ===== §2 指定話者をまとめて聴く =====
SPEAKER = "cv_0127"   # ← 監査したい話者
N       = 6           # 再生するクリップ数
MODE    = "head"      # head=esd 順の先頭 / random / long=長い順

import numpy as np, soundfile as sf
from IPython.display import display, Audio, Markdown

clips = SPK.get(SPEAKER, [])
assert clips, (
    f"★{SPEAKER} の音声がこのセッションに無い。\n"
    "  - データ配置済みセッション（eval 等）なら §1 を先に実行する\n"
    "  - さらのセッションなら §1.5 の AUDIT_SPEAKERS にこの話者を入れて実行 → §1 を再実行 → ここへ\n"
    "  - purity の worst_pair だけ聴きたい場合は §4（TSV 連携の一括聴取）が最短")
def dur(p):
    i = sf.info(str(p)); return i.frames / i.samplerate
if MODE == "random":
    rng = np.random.default_rng(0)
    pick = [clips[i] for i in rng.permutation(len(clips))[:N]]
elif MODE == "long":
    pick = sorted(clips, key=lambda x: -dur(x[0]))[:N]
else:
    pick = clips[:N]

g = GEN.get(SPEAKER, "?")
display(Markdown(f"### {SPEAKER} — 全 {len(clips)} 発話 / 性別メタ: **{g}**"))
for w, t in pick:
    display(Markdown(f"`{w.name}`（{dur(w):.1f}s）: {t}"))
    display(Audio(str(w)))


In [ ]:
# ===== §3 x-vector 近傍話者との聴き比べ（speaker_map がある場合）=====
K = 4   # 近傍を何名聴くか（各1クリップ）

assert COORD, "★speaker_map.json 未読込（§1 の MAP を設定）"
import numpy as np
me = np.array(COORD[SPEAKER])
near = sorted(((np.linalg.norm(np.array(v) - me), s) for s, v in COORD.items() if s != SPEAKER))[:K]
display(Markdown(f"### {SPEAKER}（{GEN.get(SPEAKER,'?')}）の近傍 {K} 名（マップ座標のユークリッド距離順）"))
for d, s in near:
    w, t = SPK[s][0]
    display(Markdown(f"**{s}**（{GEN.get(s,'?')} / 距離 {d:.3f}）`{w.name}`: {t}"))
    display(Audio(str(w)))
display(Markdown("※ マップは UMAP のため距離の見た目は歪む（近傍関係の参考として）"))


## §4–§5 一括監査（purity レポート連携）\n\n`speaker_purity_report.py` の TSV を直接読み、候補全員の最疑ペアを上から順に再生 → 判定を記入して JSON に保存する。**§1 まで実行してから**使う（データ未配置でも可 — 必要な wav だけ tar から選択展開する）。

In [ ]:
# ===== §4 一括聴取 — purity レポートの候補（MESSY?/SHARED?）を上から順に聴く =====
# 前提: speaker_purity_report.py の出力 TSV。どこで実行したかに応じて下の候補パスを直す。
#   （purity 実行後に `cp purity_v2.tsv /content/drive/MyDrive/Style-Bert-VITS2/` しておくと
#     セッションをまたいでもここから直接読める）
PURITY_TSV = None      # None = 下の候補から自動検出 / 明示パスも可
TYPES = ("SHARED?F0", "INTRUDER?F0", "MIX?", "MESSY?", "SHARED?")   # 聴く型（新旧 TSV 両対応。outlier も聴くなら足す）
LIMIT = None                    # 先頭 N 名だけ聴くなら数値

import subprocess, tarfile
from pathlib import Path
import numpy as np, soundfile as sf
from IPython.display import display, Audio, Markdown

_cand = [Path("/content/Style-Bert-VITS2/purity_v2.tsv"),
         Path("/content/drive/MyDrive/Style-Bert-VITS2/purity_v2.tsv"),
         Path("purity_v2.tsv")]
tsv = Path(PURITY_TSV) if PURITY_TSV else next((c for c in _cand if c.exists()), None)
assert tsv, "★purity の TSV が見つからない: PURITY_TSV を設定（Drive にコピーしておくのが楽）"

rows = []
for line in open(tsv, encoding="utf-8"):
    if line.startswith("#") or line.startswith("spk\t"):
        continue
    c = line.rstrip("\n").split("\t")
    if len(c) >= 9 and c[-1] in TYPES:   # type は常に最終列（v2/v3 両対応）
        rows.append(dict(spk=c[0], mean=float(c[2]), pair=c[4].split("|"),
                         split=c[5], across=float(c[6]),
                         f0=(c[-2] if len(c) >= 10 else "-"), typ=c[-1]))
rows = rows[:LIMIT] if LIMIT else rows
print(f"聴取対象: {len(rows)} 名（{' / '.join(TYPES)}）")

# --- worst_pair の wav を確保（配置済み → そのまま / 無ければ tar から名前指定で選択展開）---
def _wav_of(name):
    for base in ([DATA] if DATA else []) + [Path("/content/audit_data/Data/cv_r1")]:
        p = base / "wavs" / name
        if p.exists():
            return p
    return None

need = sorted({n for r in rows for n in r["pair"] if _wav_of(n) is None})
if need:
    print(f"未配置 {len(need)} 本を wavs tar から選択展開...")
    ROOT = Path("/content/audit_data"); ROOT.mkdir(exist_ok=True)
    CACHE = Path("/content/drive/MyDrive/Style-Bert-VITS2/zenodo_cache")
    WAVS_N = "cadence_cv_r1_wavs_v20260702.tar"
    src = CACHE / WAVS_N
    if not (src.exists() and src.stat().st_size > 4_000_000_000):
        src = ROOT / WAVS_N
        if not (src.exists() and src.stat().st_size > 4_000_000_000):
            subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
            subprocess.run(["aria2c", "-x16", "-s16", "-q", "-d", str(ROOT), "-o", WAVS_N,
                            f"https://zenodo.org/records/21119791/files/{WAVS_N}?download=1"], check=True)
    nd = {f"Data/cv_r1/wavs/{n}" for n in need}
    with tarfile.open(src) as t:
        for m in t:
            if m.name in nd:
                t.extract(m, ROOT, filter="data")
                nd.discard(m.name)
                if not nd:
                    break
    print("展開完了")

# --- 上から順に再生（各話者 = 最疑ペア2本）---
for i, r in enumerate(rows, 1):
    g = GEN.get(r["spk"], "?")
    display(Markdown(f"---\n### [{i}/{len(rows)}] {r['typ']} **{r['spk']}**"
                     f"（性別メタ {g} / split {r['split']} / 塊間 cos {r['across']:.3f}"
                     f" / ΔF0 {r['f0']}st / mean {r['mean']:.3f}）"))
    for n in r["pair"]:
        w = _wav_of(n)
        if w is None:
            display(Markdown(f"（{n}: 取得失敗）"))
            continue
        info = sf.info(str(w))
        display(Markdown(f"`{n}`（{info.frames/info.samplerate:.1f}s）"))
        display(Audio(str(w)))

# --- §5 に貼る判定テンプレートを出力 ---
print("\n=== §5 用テンプレート（コピーして判定を記入: shared / single / unsure）===")
print("VERDICT = {")
for r in rows:
    print(f'    "{r["spk"]}": "",   # {r["typ"]}')
print("}")


In [ ]:
# ===== §5 判定の記録 — §4 のテンプレートを貼り、判定を記入して実行 =====
# 記入値: "shared"（複数人と確認）/ "single"（同一人物。環境差など）/ "unsure"（保留）/ ""（未聴取）
VERDICT = {
    # ここに §4 の出力テンプレートを貼って記入する
}

import json, collections, time
from pathlib import Path
assert VERDICT, "★§4 の末尾に出るテンプレートを貼って判定を記入してから実行"
cnt = collections.Counter(v or "(未記入)" for v in VERDICT.values())
out = {"created": time.strftime("%Y-%m-%dT%H:%M:%S"),
       "n": len(VERDICT), "counts": dict(cnt), "verdicts": VERDICT}
dst = Path("/content/drive/MyDrive/Style-Bert-VITS2/purity_audit_verdicts.json")
try:
    dst.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding="utf-8")
    where = dst
except OSError:
    where = Path("purity_audit_verdicts.json")
    where.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding="utf-8")
print("保存:", where)
print("内訳:", dict(cnt))
print("→ データシート §6 の記載例: "
      f"「純度検査の候補 {len(VERDICT)} 名を聴取し、複数人の混入を {cnt.get('shared', 0)} 名で確認」")
